# Módulo 5 — Análise de Qualidade de Dados no Snowflake

Combinamos dados de **cadastro** e **rede elétrica** para construir análises de qualidade reais,
rodando diretamente no Snowflake.

## O que vamos ver

1. Amostra de consumidores via `tb_consumidor.sql`
2. Análise de nulos e completude do cadastro
3. Consistência de coordenadas e campos categóricos
4. Conectividade de rede: consumidor → nó → trecho
5. Agente Gemini com ferramentas Snowflake para exploração interativa

In [1]:
import sys
import os
import json
import time
import warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# --- Resolver caminhos (igual ao notebook 01) ---
cwd = Path().resolve()
candidatos = [
    cwd,
    cwd / "modulo5_snowflake_qualidade",
    cwd.parent / "modulo5_snowflake_qualidade",
]
modulo5_dir = next((p for p in candidatos if (p / "snowflake_db.py").exists()), None)
if modulo5_dir is None:
    raise FileNotFoundError("snowflake_db.py não encontrado")

if str(modulo5_dir) not in sys.path:
    sys.path.insert(0, str(modulo5_dir))

for env_candidate in [cwd / ".env", cwd.parent / ".env", modulo5_dir.parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(dotenv_path=env_candidate, override=True)
        print(f".env carregado: {env_candidate}")
        break

from snowflake_db import SNOWFLAKE_CONFIG, read_sql_file, read_sql_to_dataframe
import pandas as pd

project_root = modulo5_dir.parent
print(f"Projeto     : {project_root}")
print(f"Conta       : {SNOWFLAKE_CONFIG['account']}")
print(f"Database    : {SNOWFLAKE_CONFIG['database']}")

.env carregado: /home/vinicius/projects/t2t/python-learning/.env


Projeto     : /home/vinicius/projects/t2t/python-learning
Conta       : khb56279.us-east-1
Database    : EQTLINFO_HML


## 1. Amostra de consumidores

Usamos `queries/tb_consumidor.sql` para trazer 1 000 registros aleatórios de
`EQTLINFO_PRD.EQTL_PA.TAB_CADASTRO` com JOIN na tabela operacional.

O `ORDER BY RANDOM()` garante que cada execução traz uma amostra diferente —
ideal para análise exploratória sem viés de seleção.

In [2]:
sql_consumidor = project_root / "queries" / "tb_consumidor.sql"

print("Carregando amostra de consumidores...")
df = read_sql_to_dataframe(sql_consumidor)

print(f"Registros retornados : {len(df)}")
print(f"Colunas              : {list(df.columns)}")
df.head(3)

INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...


INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39


INFO: Connecting to GLOBAL Snowflake domain


Carregando amostra de consumidores...


INFO: Conexão estabelecida com sucesso.


INFO: Conexão encerrada.


Registros retornados : 20
Colunas              : ['CR_NUMERO', 'INSTALACAO_ID', 'CONTRATO', 'CONTA_CONTRATO', 'MEDIDOR_ID', 'TIPO_LOCAL_CONSUMO', 'GENERO', 'DATA_NASCIMENTO', 'ESTADO_CIVIL', 'LATITUDE', 'LONGITUDE', 'DATA_INSTALACAO_MEDIDOR']


,CR_NUMERO,INSTALACAO_ID,CONTRATO,CONTA_CONTRATO,MEDIDOR_ID,TIPO_LOCAL_CONSUMO,GENERO,DATA_NASCIMENTO,ESTADO_CIVIL,LATITUDE,LONGITUDE,DATA_INSTALACAO_MEDIDOR
0,2001005086,3117679.00,504932092.0,3.023363e+09,NaN,Casa,Feminino,1991-07-09,Outros,None,None,None
1,0011252850,None,11252850.0,1.125285e+07,1.702301e+10,Casa,Masculino,1963-10-31,None,-1.3040191,-48.4737808,2025-09-22
2,0106968969,None,503963018.0,1.069690e+08,1.201067e+10,Casa,Feminino,1970-08-20,Divorciado(a),-2.4779197,-54.7148847,2021-04-05


## 2. Análise de nulos e completude

Calculamos o percentual de nulos por coluna e classificamos por criticidade.

| Status | Critério |
|--------|----------|
| CRÍTICO | Campo obrigatório com qualquer nulo, ou qualquer campo > 20% nulo |
| ATENÇÃO | > 5% de nulos |
| OK | ≤ 5% de nulos |

In [3]:
CAMPOS_CRITICOS = {"CONTRATO", "MEDIDOR_ID", "LATITUDE", "LONGITUDE"}

colunas = [c for c in df.columns if c != "CR_NUMERO"]

df_qualidade = pd.DataFrame([
    {
        "CAMPO": c,
        "PREENCHIDOS": df[c].notna().sum(),
        "NULOS": df[c].isnull().sum(),
        "PCT_NULO": round(df[c].isnull().sum() / len(df) * 100, 1),
    }
    for c in colunas
])

def _status(row):
    if row["CAMPO"] in CAMPOS_CRITICOS and row["NULOS"] > 0:
        return "CRÍTICO"
    if row["PCT_NULO"] > 20:
        return "CRÍTICO"
    if row["PCT_NULO"] > 5:
        return "ATENÇÃO"
    return "OK"

df_qualidade["STATUS"] = df_qualidade.apply(_status, axis=1)
df_qualidade = df_qualidade.sort_values("PCT_NULO", ascending=False).reset_index(drop=True)

score = round(df_qualidade["PREENCHIDOS"].sum() / (len(df) * len(colunas)) * 100, 1)
print(f"Score geral de completude: {score}%")
print(f"Campos CRÍTICOS : {(df_qualidade['STATUS'] == 'CRÍTICO').sum()}")
print(f"Campos ATENÇÃO  : {(df_qualidade['STATUS'] == 'ATENÇÃO').sum()}")
print()
print(df_qualidade.to_string(index=False))

Score geral de completude: 70.5%
Campos CRÍTICOS : 10
Campos ATENÇÃO  : 0

                  CAMPO  PREENCHIDOS  NULOS  PCT_NULO  STATUS
          INSTALACAO_ID            3     17      85.0 CRÍTICO
           ESTADO_CIVIL            8     12      60.0 CRÍTICO
             MEDIDOR_ID           13      7      35.0 CRÍTICO
DATA_INSTALACAO_MEDIDOR           14      6      30.0 CRÍTICO
        DATA_NASCIMENTO           14      6      30.0 CRÍTICO
         CONTA_CONTRATO           15      5      25.0 CRÍTICO
               CONTRATO           15      5      25.0 CRÍTICO
                 GENERO           15      5      25.0 CRÍTICO
               LATITUDE           19      1       5.0 CRÍTICO
              LONGITUDE           19      1       5.0 CRÍTICO
     TIPO_LOCAL_CONSUMO           20      0       0.0      OK


## 3. Consistência dos dados

Verificações além de nulos — os valores fazem sentido?

- **Coordenadas** dentro dos limites geográficos do Pará
- **Data de nascimento** em intervalo plausível
- **Categorias** esperadas em campos de tipo/gênero/estado civil

In [4]:
# Limites aproximados do estado do Pará
LAT_MIN, LAT_MAX = -10.0, 2.5
LON_MIN, LON_MAX = -54.0, -46.0

print("=== COORDENADAS ===")
df_coords = df[df["LATITUDE"].notna() & df["LONGITUDE"].notna()]
fora_pa = df_coords[
    (df_coords["LATITUDE"].astype(float) < LAT_MIN) |
    (df_coords["LATITUDE"].astype(float) > LAT_MAX) |
    (df_coords["LONGITUDE"].astype(float) < LON_MIN) |
    (df_coords["LONGITUDE"].astype(float) > LON_MAX)
]
print(f"  Com coordenadas    : {len(df_coords)}/{len(df)}")
print(f"  Fora dos limites PA: {len(fora_pa)} ({round(len(fora_pa)/max(len(df_coords),1)*100,1)}%)")

print("\n=== DATA DE NASCIMENTO ===")
df["DATA_NASCIMENTO"] = pd.to_datetime(df["DATA_NASCIMENTO"], errors="coerce")
df_nasc = df[df["DATA_NASCIMENTO"].notna()]
invalidos = df_nasc[
    (df_nasc["DATA_NASCIMENTO"].dt.year < 1900) |
    (df_nasc["DATA_NASCIMENTO"].dt.year > 2010)
]
print(f"  Com data nascimento: {len(df_nasc)}/{len(df)}")
print(f"  Fora do range      : {len(invalidos)}")
if not df_nasc.empty:
    print(f"  Idade média aprox. : {2025 - int(df_nasc['DATA_NASCIMENTO'].dt.year.mean())} anos")

print("\n=== DISTRIBUIÇÃO DE CATEGORIAS ===")
for col in ["TIPO_LOCAL_CONSUMO", "GENERO", "ESTADO_CIVIL"]:
    if col in df.columns:
        dist = df[col].value_counts(dropna=False)
        print(f"\n  {col}:")
        for val, cnt in dist.items():
            pct = round(cnt / len(df) * 100, 1)
            print(f"    {str(val):<25} {cnt:>5} ({pct}%)")

=== COORDENADAS ===
  Com coordenadas    : 19/20
  Fora dos limites PA: 2 (10.5%)

=== DATA DE NASCIMENTO ===
  Com data nascimento: 14/20
  Fora do range      : 0
  Idade média aprox. : 59 anos

=== DISTRIBUIÇÃO DE CATEGORIAS ===

  TIPO_LOCAL_CONSUMO:
    Casa                         18 (90.0%)
    Indefinido                    2 (10.0%)

  GENERO:
    Masculino                     7 (35.0%)
    Feminino                      6 (30.0%)
    None                          5 (25.0%)
    Indefinido                    2 (10.0%)

  ESTADO_CIVIL:
    None                         12 (60.0%)
    Outros                        5 (25.0%)
    Solteiro(a)                   2 (10.0%)
    Divorciado(a)                 1 (5.0%)


## 4. Conectividade de rede

A tabela `EQTLINFO_RAW.MAPA_PA.CONSUMIDOR` liga o consumidor à topologia da rede elétrica:

```
CR_NUMERO → NO_ID (nó na rede)
NO_ID → TRECHO_DE_REDE (segmentos conectados ao nó)
```

Um consumidor sem `NO_ID` está cadastrado no sistema comercial mas não possui
vínculo com a rede georreferenciada — possível problema de integração entre sistemas.

In [5]:
# Pegar os CR_NUMEROs da amostra e verificar quais têm vínculo na MAPA_PA.CONSUMIDOR
# CR_NUMERO em MAPA_PA é numérico (sem zeros à esquerda e sem ".00")
crs_amostra = df["CR_NUMERO"].dropna().head(200).tolist()

# Converter para inteiro removendo zeros à esquerda
crs_numericos = [str(int(cr.lstrip("0") or "0")) for cr in crs_amostra]
crs_in = ", ".join(crs_numericos)

query_rede = f"""
SELECT
    C.CR_NUMERO,
    C.NO_ID,
    C.MSLINK_TR,
    C.MSLINK_INS,
    COUNT(T.MSLINK)             AS QTD_TRECHOS_CONECTADOS,
    -- TRC_COMPRIMENTO: comprimento do segmento NO_ID_DE → NO_ID_PA (metros)
    SUM(T.TRC_COMPRIMENTO)      AS COMPRIMENTO_TOTAL_M,
    AVG(T.TRC_COMPRIMENTO)      AS COMPRIMENTO_MEDIO_M,
    -- TRC_NIVEL: nível do trecho — indica se a rede é ascendente ou descendente na hierarquia
    MIN(T.TRC_NIVEL)            AS NIVEL_MIN,
    MAX(T.TRC_NIVEL)            AS NIVEL_MAX
FROM EQTLINFO_RAW.MAPA_PA.CONSUMIDOR C
LEFT JOIN EQTLINFO_RAW.MAPA_PA.TRECHO_DE_REDE T
    ON T.NO_ID_DE = C.NO_ID OR T.NO_ID_PA = C.NO_ID
WHERE C.CR_NUMERO IN ({crs_in})
GROUP BY C.CR_NUMERO, C.NO_ID, C.MSLINK_TR, C.MSLINK_INS
"""

print("Verificando conectividade na rede elétrica...")
df_rede = read_sql_to_dataframe(query_rede)

print(f"Consumidores na amostra          : {len(crs_amostra)}")
print(f"Encontrados na MAPA_PA           : {len(df_rede)}")
print(f"Sem vínculo na rede (mapa)       : {len(crs_amostra) - len(df_rede)}")

if not df_rede.empty:
    com_no = df_rede["NO_ID"].notna().sum()
    print(f"\nCom NO_ID (nó georreferenciado)  : {com_no}/{len(df_rede)}")
    print(f"Com MSLINK_TR                    : {df_rede['MSLINK_TR'].notna().sum()}")
    print(f"Com MSLINK_INS                   : {df_rede['MSLINK_INS'].notna().sum()}")

    # Comprimento
    print(f"\n--- TRC_COMPRIMENTO ---")
    print(f"Comprimento total (soma, m)      : {df_rede['COMPRIMENTO_TOTAL_M'].sum():.1f}")
    print(f"Comprimento médio por nó (m)     : {df_rede['COMPRIMENTO_MEDIO_M'].mean():.1f}")
    print(f"Trechos conectados (média)       : {df_rede['QTD_TRECHOS_CONECTADOS'].mean():.1f}")
    print(f"Trechos conectados (máx)         : {df_rede['QTD_TRECHOS_CONECTADOS'].max()}")

    # Nível
    print(f"\n--- TRC_NIVEL ---")
    if df_rede["NIVEL_MIN"].notna().any():
        print(f"Nível mínimo encontrado          : {df_rede['NIVEL_MIN'].min()}")
        print(f"Nível máximo encontrado          : {df_rede['NIVEL_MAX'].max()}")
        dist_nivel = df_rede["NIVEL_MIN"].value_counts().sort_index()
        print(f"Distribuição por nível (NIVEL_MIN):")
        for nivel, cnt in dist_nivel.items():
            print(f"  Nível {nivel}: {cnt} consumidores")
    else:
        print("  TRC_NIVEL: sem dados nesta amostra")

    print()
    print(df_rede[["CR_NUMERO", "NO_ID", "QTD_TRECHOS_CONECTADOS",
                   "COMPRIMENTO_TOTAL_M", "NIVEL_MIN", "NIVEL_MAX"]].head(10).to_string(index=False))

INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...


INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39


INFO: Connecting to GLOBAL Snowflake domain


Verificando conectividade na rede elétrica...


INFO: Conexão estabelecida com sucesso.


INFO: Conexão encerrada.


Consumidores na amostra          : 20
Encontrados na MAPA_PA           : 18
Sem vínculo na rede (mapa)       : 2

Com NO_ID (nó georreferenciado)  : 18/18
Com MSLINK_TR                    : 16
Com MSLINK_INS                   : 2

--- TRC_COMPRIMENTO ---
Comprimento total (soma, m)      : 1049.9
Comprimento médio por nó (m)     : 34.4
Trechos conectados (média)       : 1.7
Trechos conectados (máx)         : 3

--- TRC_NIVEL ---
Nível mínimo encontrado          : 1
Nível máximo encontrado          : 1370
Distribuição por nível (NIVEL_MIN):
  Nível 1: 4 consumidores
  Nível 2: 4 consumidores
  Nível 3: 1 consumidores
  Nível 4: 2 consumidores
  Nível 5: 2 consumidores
  Nível 6: 1 consumidores
  Nível 8: 1 consumidores
  Nível 10: 1 consumidores
  Nível 360: 1 consumidores
  Nível 1370: 1 consumidores

    CR_NUMERO       NO_ID  QTD_TRECHOS_CONECTADOS COMPRIMENTO_TOTAL_M  NIVEL_MIN  NIVEL_MAX
   7855486.00 12178224.00                       2               67.72          2          3
2001

## 5. Agente Gemini com ferramentas Snowflake

Conectamos o Gemini a ferramentas reais de consulta no Snowflake.
O agente decide autonomamente quais queries executar para responder perguntas sobre qualidade de dados.

```
Pergunta do usuário
      ↓
   Gemini decide qual ferramenta usar
      ↓
   Ferramenta executa query no Snowflake
      ↓
   Resultado retorna ao Gemini
      ↓
   Gemini formula a resposta final
```

In [6]:
try:
    from google import genai
    from google.genai import types
    print("google-genai: OK")
except ImportError:
    print("Instale com: uv add google-genai")

client = genai.Client()
MODEL = "gemini-2.0-flash"

sql_por_cr = project_root / "queries" / "tb_consumidor_por_cr.sql"

# ---------------------------------------------------------------------------
# Ferramentas disponíveis para o agente
# ---------------------------------------------------------------------------

def buscar_consumidor(cr_numero: str) -> dict:
    """Busca dados cadastrais de um consumidor pelo CR_NUMERO."""
    df_cr = read_sql_to_dataframe(sql_por_cr, cr_numero=cr_numero)
    if df_cr.empty:
        return {"encontrado": False, "cr_numero": cr_numero}
    row = df_cr.iloc[0].to_dict()
    return {"encontrado": True, **{k: str(v) for k, v in row.items()}}


def analisar_qualidade_amostra(limite: int = 100) -> dict:
    """Analisa completude de campos obrigatórios em uma amostra de consumidores."""
    query = f"""
    SELECT
        COUNT(*)                                                           AS TOTAL,
        SUM(CASE WHEN CONTRATO IS NULL        THEN 1 ELSE 0 END)          AS SEM_CONTRATO,
        SUM(CASE WHEN MEDIDOR_ID IS NULL      THEN 1 ELSE 0 END)          AS SEM_MEDIDOR,
        SUM(CASE WHEN LATITUDE IS NULL        THEN 1 ELSE 0 END)          AS SEM_LATITUDE,
        SUM(CASE WHEN DATA_NASCIMENTO IS NULL THEN 1 ELSE 0 END)          AS SEM_NASC,
        SUM(CASE WHEN INSTALACAO_ID IS NULL   THEN 1 ELSE 0 END)          AS SEM_INSTALACAO
    FROM (
        SELECT
            TAB_CADASTRO.INSTALACAO                                AS CR_NUMERO,
            CONSUMIDOR.INSTALACAO_ID,
            TRY_CAST(TAB_CADASTRO.CONTRATO  AS INTEGER)            AS CONTRATO,
            TRY_CAST(TAB_CADASTRO.MEDIDOR   AS INTEGER)            AS MEDIDOR_ID,
            TAB_CADASTRO.DATA_NASCIMENTO,
            TAB_CADASTRO.LATITUDE
        FROM (
            SELECT * FROM EQTLINFO_PRD.EQTL_PA.TAB_CADASTRO
            ORDER BY RANDOM() LIMIT {limite}
        ) TAB_CADASTRO
        LEFT JOIN EQTLINFO_RAW.OPER_PA.CONSUMIDOR CONSUMIDOR
            ON TAB_CADASTRO.INSTALACAO = LPAD(
                REGEXP_REPLACE(TO_VARCHAR(CONSUMIDOR.CR_NUMERO), '[^0-9]', ''),
                LENGTH(TAB_CADASTRO.INSTALACAO), '0'
            )
    )
    """
    df_q = read_sql_to_dataframe(query)
    if df_q.empty:
        return {"erro": "sem resultado"}
    r = df_q.iloc[0]
    total = int(r["TOTAL"])
    campos = ["SEM_CONTRATO", "SEM_MEDIDOR", "SEM_LATITUDE", "SEM_NASC", "SEM_INSTALACAO"]
    return {
        "total_analisado": total,
        **{c.lower(): int(r[c]) for c in campos},
        "pct_criticos_preenchidos": round(
            (1 - (int(r["SEM_CONTRATO"]) + int(r["SEM_MEDIDOR"]) + int(r["SEM_LATITUDE"])) / (3 * total)) * 100, 1
        ),
    }


def buscar_rede_consumidor(cr_numero: str) -> dict:
    """Busca o nó de rede, trechos elétricos, comprimento e nível conectados ao consumidor.

    TRC_COMPRIMENTO: comprimento do segmento NO_ID_DE → NO_ID_PA em metros.
    TRC_NIVEL: nível hierárquico do trecho na rede (ascendente ou não).
    """
    cr_num = str(int(cr_numero.lstrip("0") or "0"))
    query = f"""
    SELECT
        C.CR_NUMERO,
        C.NO_ID,
        C.MSLINK_TR,
        C.MSLINK_INS,
        COUNT(T.MSLINK)             AS QTD_TRECHOS,
        -- comprimento total dos trechos conectados ao nó do consumidor (metros)
        SUM(T.TRC_COMPRIMENTO)      AS COMPRIMENTO_TOTAL_M,
        AVG(T.TRC_COMPRIMENTO)      AS COMPRIMENTO_MEDIO_M,
        -- nível hierárquico: menor = mais próximo da subestação
        MIN(T.TRC_NIVEL)            AS NIVEL_MIN,
        MAX(T.TRC_NIVEL)            AS NIVEL_MAX
    FROM EQTLINFO_RAW.MAPA_PA.CONSUMIDOR C
    LEFT JOIN EQTLINFO_RAW.MAPA_PA.TRECHO_DE_REDE T
        ON T.NO_ID_DE = C.NO_ID OR T.NO_ID_PA = C.NO_ID
    WHERE C.CR_NUMERO = {cr_num}
    GROUP BY C.CR_NUMERO, C.NO_ID, C.MSLINK_TR, C.MSLINK_INS
    """
    df_r = read_sql_to_dataframe(query)
    if df_r.empty:
        return {"encontrado": False, "cr_numero": cr_numero, "mensagem": "CR não encontrado na MAPA_PA.CONSUMIDOR"}
    r = df_r.iloc[0]
    return {
        "encontrado": True,
        "cr_numero": cr_numero,
        "no_id": str(r.get("NO_ID", "")),
        "mslink_tr": str(r.get("MSLINK_TR", "")),
        "mslink_ins": str(r.get("MSLINK_INS", "")),
        "qtd_trechos_conectados": int(r.get("QTD_TRECHOS", 0)),
        "comprimento_total_m": float(r.get("COMPRIMENTO_TOTAL_M") or 0),
        "comprimento_medio_m": float(r.get("COMPRIMENTO_MEDIO_M") or 0),
        "nivel_min": None if r.get("NIVEL_MIN") is None else int(r["NIVEL_MIN"]),
        "nivel_max": None if r.get("NIVEL_MAX") is None else int(r["NIVEL_MAX"]),
    }


TOOL_MAP = {
    "buscar_consumidor": buscar_consumidor,
    "analisar_qualidade_amostra": analisar_qualidade_amostra,
    "buscar_rede_consumidor": buscar_rede_consumidor,
}

# ---------------------------------------------------------------------------
# Declarações de função para o Gemini
# ---------------------------------------------------------------------------

TOOLS = [types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="buscar_consumidor",
        description="Busca dados cadastrais completos de um consumidor no Snowflake pelo CR_NUMERO.",
        parameters=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "cr_numero": types.Schema(
                    type=types.Type.STRING,
                    description="CR_NUMERO com zeros à esquerda, ex: '0002997827'",
                ),
            },
            required=["cr_numero"],
        ),
    ),
    types.FunctionDeclaration(
        name="analisar_qualidade_amostra",
        description="Analisa completude dos campos obrigatórios em uma amostra aleatória de consumidores.",
        parameters=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "limite": types.Schema(
                    type=types.Type.INTEGER,
                    description="Quantidade de registros a amostrar (padrão 100, máximo recomendado 500)",
                ),
            },
            required=[],
        ),
    ),
    types.FunctionDeclaration(
        name="buscar_rede_consumidor",
        description=(
            "Verifica a conectividade de um consumidor na rede elétrica georreferenciada. "
            "Retorna: nó (NO_ID), quantidade de trechos conectados, comprimento total e médio dos trechos (metros), "
            "e TRC_NIVEL (nível hierárquico do trecho — menor = mais próximo da subestação)."
        ),
        parameters=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "cr_numero": types.Schema(
                    type=types.Type.STRING,
                    description="CR_NUMERO do consumidor (com ou sem zeros à esquerda)",
                ),
            },
            required=["cr_numero"],
        ),
    ),
])]

print(f"Modelo    : {MODEL}")
print(f"Ferramentas disponíveis: {list(TOOL_MAP.keys())}")

google-genai: OK
Modelo    : gemini-2.0-flash
Ferramentas disponíveis: ['buscar_consumidor', 'analisar_qualidade_amostra', 'buscar_rede_consumidor']


In [7]:
def rodar_agente(pergunta: str) -> str:
    """Executa o agente Gemini com ferramentas Snowflake até obter resposta final."""

    SYSTEM = (
        "Você é um analista de dados especializado em qualidade de dados do setor elétrico brasileiro. "
        "Use as ferramentas disponíveis para buscar dados reais no Snowflake antes de responder. "
        "Seja objetivo e foque em insights acionáveis para o negócio."
    )

    messages = [types.Content(role="user", parts=[types.Part(text=pergunta)])]

    print(f"Pergunta: {pergunta}")
    print("-" * 60)

    for _ in range(8):
        response = client.models.generate_content(
            model=MODEL,
            contents=messages,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM,
                tools=TOOLS,
                temperature=0.1,
            ),
        )

        candidate = response.candidates[0]
        tool_calls = [
            p for p in candidate.content.parts
            if hasattr(p, "function_call") and p.function_call
        ]

        # Sem chamada de ferramenta → resposta final
        if not tool_calls:
            texto = "".join(p.text for p in candidate.content.parts if hasattr(p, "text"))
            print(f"Resposta:\n{texto}")
            return texto

        # Executar ferramentas
        messages.append(candidate.content)
        tool_results = []

        for part in tool_calls:
            fc = part.function_call
            print(f"  → {fc.name}({dict(fc.args)})")
            try:
                resultado = TOOL_MAP[fc.name](**dict(fc.args))
            except Exception as e:
                resultado = {"erro": str(e)}
            preview = json.dumps(resultado, default=str)[:300]
            print(f"    {preview}{'...' if len(json.dumps(resultado, default=str)) > 300 else ''}")

            tool_results.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        name=fc.name,
                        response={"result": resultado},
                    )
                )
            )

        messages.append(types.Content(role="user", parts=tool_results))
        time.sleep(0.3)

    return "Limite de rodadas atingido."


# Exemplo 1 — visão geral de qualidade
rodar_agente(
    "Analise a qualidade dos dados de uma amostra de 200 consumidores. "
    "Quais são os principais problemas e o que você recomenda priorizar?"
)

Pergunta: Analise a qualidade dos dados de uma amostra de 200 consumidores. Quais são os principais problemas e o que você recomenda priorizar?
------------------------------------------------------------


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...


INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39


INFO: Connecting to GLOBAL Snowflake domain


...

INFO: Conexão encerrada.


    {"total_analisado": 200, "sem_contrato": 48, "sem_medidor": 61, "sem_latitude": 5, "sem_nasc": 52, "sem_instalacao": 156, "pct_criticos_preenchidos": 81.0}


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


Resposta:
A análise de qualidade de dados em uma amostra de 200 consumidores revelou os seguintes pontos críticos:

*   **Ausência de Instalação:** 156 consumidores sem informação de instalação.
*   **Ausência de Medidor:** 61 consumidores sem informação de medidor.
*   **Ausência de Data de Nascimento:** 52 consumidores sem data de nascimento.
*   **Ausência de Contrato:** 48 consumidores sem informação de contrato.
*   **Ausência de Latitude:** 5 consumidores sem informação de latitude.
*   **Percentual de campos críticos preenchidos:** 81%

**Recomendações:**

1.  **Priorizar Instalação:** A ausência de dados de instalação é o problema mais prevalente. Investigar a causa raiz dessa falta de informação (problemas de coleta, migração de dados, etc.) e implementar um processo para garantir a captura e manutenção desses dados.
2.  **Medidor e Contrato:** A ausência de dados de medidor e contrato também é alta. Validar os processos de cadastro e instalação de novos consumidores, garantin

'A análise de qualidade de dados em uma amostra de 200 consumidores revelou os seguintes pontos críticos:\n\n*   **Ausência de Instalação:** 156 consumidores sem informação de instalação.\n*   **Ausência de Medidor:** 61 consumidores sem informação de medidor.\n*   **Ausência de Data de Nascimento:** 52 consumidores sem data de nascimento.\n*   **Ausência de Contrato:** 48 consumidores sem informação de contrato.\n*   **Ausência de Latitude:** 5 consumidores sem informação de latitude.\n*   **Percentual de campos críticos preenchidos:** 81%\n\n**Recomendações:**\n\n1.  **Priorizar Instalação:** A ausência de dados de instalação é o problema mais prevalente. Investigar a causa raiz dessa falta de informação (problemas de coleta, migração de dados, etc.) e implementar um processo para garantir a captura e manutenção desses dados.\n2.  **Medidor e Contrato:** A ausência de dados de medidor e contrato também é alta. Validar os processos de cadastro e instalação de novos consumidores, garan

In [8]:
# Exemplo 2 — análise combinada de cadastro + rede para um CR específico
rodar_agente(
    "Busque o consumidor CR 0002997827. "
    "Verifique também se ele está conectado na rede elétrica. "
    "Me dê um diagnóstico completo desse cadastro."
)

Pergunta: Busque o consumidor CR 0002997827. Verifique também se ele está conectado na rede elétrica. Me dê um diagnóstico completo desse cadastro.
------------------------------------------------------------


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...


INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39


INFO: Connecting to GLOBAL Snowflake domain


...

INFO: Conexão encerrada.


    {"encontrado": true, "cr_numero": "0002997827", "no_id": "7792709.00", "mslink_tr": "9114623", "mslink_ins": "None", "qtd_trechos_conectados": 2, "comprimento_total_m": 69.77, "comprimento_medio_m": 34.885, "nivel_min": 3, "nivel_max": 4}


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


Resposta:
O consumidor CR 0002997827 foi encontrado no cadastro. Ele é do gênero feminino, solteira, nasceu em 07/03/1967 e possui um medidor instalado em 02/12/2015. A instalação está localizada nas coordenadas geográficas (latitude: -1.3465397, longitude: -48.4693652). O tipo de local de consumo é "Casa".

Adicionalmente, o consumidor está conectado à rede elétrica através do nó 7792709.00, com 2 trechos conectados, totalizando 69.77 metros de cabos. O nível hierárquico do trecho (TRC_NIVEL) varia entre 3 e 4.


'O consumidor CR 0002997827 foi encontrado no cadastro. Ele é do gênero feminino, solteira, nasceu em 07/03/1967 e possui um medidor instalado em 02/12/2015. A instalação está localizada nas coordenadas geográficas (latitude: -1.3465397, longitude: -48.4693652). O tipo de local de consumo é "Casa".\n\nAdicionalmente, o consumidor está conectado à rede elétrica através do nó 7792709.00, com 2 trechos conectados, totalizando 69.77 metros de cabos. O nível hierárquico do trecho (TRC_NIVEL) varia entre 3 e 4.'

In [9]:
# Exemplo 3 — iteração livre: você pergunta, o agente busca e responde
# Troque o texto abaixo pela sua pergunta

rodar_agente(
    "Qual é a diferença de qualidade entre uma amostra de 50 e uma amostra de 300 consumidores?"
)

Pergunta: Qual é a diferença de qualidade entre uma amostra de 50 e uma amostra de 300 consumidores?
------------------------------------------------------------


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...


INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39


INFO: Connecting to GLOBAL Snowflake domain


...

INFO: Conexão encerrada.


    {"total_analisado": 300, "sem_contrato": 71, "sem_medidor": 82, "sem_latitude": 11, "sem_nasc": 80, "sem_instalacao": 216, "pct_criticos_preenchidos": 81.8}


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


Resposta:
A amostra de 50 consumidores tem 85.3% dos campos críticos preenchidos, enquanto a amostra de 300 consumidores tem 81.8%. Isso sugere que a amostra menor pode ter uma qualidade ligeiramente superior em termos de preenchimento de campos críticos. No entanto, a amostra maior fornece uma visão mais abrangente da qualidade dos dados, revelando um número maior de registros com dados faltantes em cada categoria (contrato, instalação, latitude, medidor, data de nascimento). A escolha entre as amostras depende do objetivo da análise: se o foco é identificar rapidamente problemas críticos, a amostra menor pode ser suficiente; se o objetivo é ter uma visão mais completa e precisa da qualidade dos dados, a amostra maior é mais adequada.


'A amostra de 50 consumidores tem 85.3% dos campos críticos preenchidos, enquanto a amostra de 300 consumidores tem 81.8%. Isso sugere que a amostra menor pode ter uma qualidade ligeiramente superior em termos de preenchimento de campos críticos. No entanto, a amostra maior fornece uma visão mais abrangente da qualidade dos dados, revelando um número maior de registros com dados faltantes em cada categoria (contrato, instalação, latitude, medidor, data de nascimento). A escolha entre as amostras depende do objetivo da análise: se o foco é identificar rapidamente problemas críticos, a amostra menor pode ser suficiente; se o objetivo é ter uma visão mais completa e precisa da qualidade dos dados, a amostra maior é mais adequada.'